# Full Fine-Tuning: EN→FR pretraining then EN→DE fine-tuning

In [ ]:

import sacrebleu

import os
import gc
import re
import math
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tqdm
import functools
from torch import optim
from torch.utils.data import TensorDataset, DataLoader, random_split, Subset
from torch.cuda.amp import GradScaler

from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
# from datasets import Dataset as HFDataset, DatasetDict
from torch.utils.data import Dataset, DataLoader, random_split  


SEED = 10701
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
CFG = {
    "en_fr_path":            "/home/ubuntu/10701-project/en-fr.csv",
    "juridia_train":         "/home/ubuntu/10701-project/en-fr.csv",
    "pretrain_sample_frac":  1.0,
    "finetune_sample_frac":  1.0,
    "vocab_size":            32000,   # shared BPE vocab (src+tgt together)
    "max_len":               128,
    "d_model":               256,
    "n_heads":               8,
    "d_k":                   32,
    "d_v":                   32,
    "d_ff":                  512,
    "n_layers":              4,
    "batch_size":            384,
    "lr_pretrain":           3e-4,
    "lr_finetune":           5e-4,
    "pretrain_epochs":       1,
    "finetune_epochs":       3,
    "train_split":           0.95,
    "num_workers":           4 if torch.cuda.is_available() else 0,
    "lora_r":                8,
    "lora_alpha":            16,
    "lora_dropout":          0.1,
    "adapter_bottleneck":    64,
    "adapter_dropout":       0.1,
    "lr_adapter":            5e-4,
    "adapter_epochs":        3,
    "tokenizer_path":        "/home/ubuntu/10701-project/bpe_tokenizer.json",
    "cache_dir":             "/home/ubuntu/10701-project/hf_cache",
}
CFG["grad_accum_steps"] = max(1, CFG["batch_size"] // CFG["batch_size"])



In [ ]:
gc.collect()
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'([^\w\s])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def load_csv_as_src_tgt(path, src_col, tgt_col, sample_frac=1.0):
    df = pd.read_csv(path, usecols=[src_col, tgt_col])
    df = df.dropna().drop_duplicates().reset_index(drop=True)
    if sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=SEED).reset_index(drop=True)
    df[src_col] = df[src_col].apply(clean_text)
    df[tgt_col] = df[tgt_col].apply(clean_text)
    return df.rename(columns={src_col: "src", tgt_col: "tgt"})

print("Loading pretrain data...")
pretrain_df = load_csv_as_src_tgt(
    CFG["en_fr_path"], "fr", "en", CFG["pretrain_sample_frac"])
print("Loading finetune data...")
ft_df = load_csv_as_src_tgt(
    CFG["juridia_train"], "fr", "en", CFG["finetune_sample_frac"])

print("finished loading finetuneh data")
finetune_train_df, finetune_test_df = train_test_split(
    ft_df, test_size=0.1, random_state=SEED, shuffle=True)
finetune_train_df = finetune_train_df.reset_index(drop=True)
finetune_test_df  = finetune_test_df.reset_index(drop=True)

print(f"Pretrain: {pretrain_df.shape}  |  FT train: {finetune_train_df.shape}  |  FT test: {finetune_test_df.shape}")

In [ ]:
def sentence_iterator(dfs, cols): 
    for df in dfs: 
        for col in cols: 
            for text in df[col]:
                yield str(text)

def train_or_load_tokenizer(cfg, dfs) -> Tokenizer:
    if os.path.exists(cfg["tokenizer_path"]")
        return Tokenizer.from_file(cfg["tokenizer_path"])

    print("Training BPE tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=cfg["vocab_size"],
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
        min_frequency=2,
        show_progress=True,
    )
    tokenizer.train_from_iterator(
        sentence_iterator(dfs, ["src", "tgt"]),
        trainer=trainer,
        length=sum(len(df) * 2 for df in dfs),  # hint for progress bar
    )

    # Wrap every sequence with <bos> ... <eos> automatically
    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    tokenizer.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )
    tokenizer.enable_padding(
        pad_id=tokenizer.token_to_id("<pad>"),
        pad_token="<pad>",
        length=cfg["max_len"],
    )
    tokenizer.enable_truncation(max_length=cfg["max_len"])

    tokenizer.save(cfg["tokenizer_path"])
    print(f"Tokenizer saved → {cfg['tokenizer_path']}")
    return tokenizer

tokenizer = train_or_load_tokenizer(
    CFG, [pretrain_df, finetune_train_df])

PAD_ID = tokenizer.token_to_id("<pad>")
VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"Vocab size: {VOCAB_SIZE}  |  PAD id: {PAD_ID}")
gc.collect()

In [ ]:
tokenizer.no_padding()
tokenizer.enable_truncation(max_length=CFG["max_len"])

class LazyTranslationDataset(Dataset):
    """
    Stores raw (src, tgt) string lists and tokenizes on __getitem__.
    No pre-built token tensors in memory — RAM cost is string data only.
    """
    def __init__(self, src_series, tgt_series, tokenizer):
        self.srcs      = src_series.tolist()
        self.tgts      = tgt_series.tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.srcs)
    def __getitem__(self, idx):
        if isinstance(idx, list):
            return [self._get_single(i) for i in idx]
        return self._get_single(int(idx))
    def _get_single(self, idx):
        idx = int(idx)
        src_ids = self.tokenizer.encode(self.srcs[idx]).ids
        tgt_ids = self.tokenizer.encode(self.tgts[idx]).ids
        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_ids, dtype=torch.long),
        )

def collate_fn_translation(batch, pad_id):
    """
    Pad a batch of variable-length sequences to the longest in that batch.
    Uses a named function (not a lambda) so it is picklable with
    num_workers > 0 via multiprocessing.
    """
    src_list, tgt_list = zip(*batch)
    max_src = max(s.size(0) for s in src_list)
    max_tgt = max(t.size(0) for t in tgt_list)

    src_padded = torch.full((len(batch), max_src), pad_id, dtype=torch.long)
    tgt_padded = torch.full((len(batch), max_tgt), pad_id, dtype=torch.long)

    for i, (src, tgt) in enumerate(batch):
        src_padded[i, :src.size(0)] = src
        tgt_padded[i, :tgt.size(0)] = tgt

    return src_padded, tgt_padded


# functools.partial creates a picklable callable — safe for multiprocessing
_collate = functools.partial(collate_fn_translation, pad_id=PAD_ID)


def make_lazy_loaders(df, tokenizer, train_split, batch_size, num_workers):
    full_ds = LazyTranslationDataset(df["src"], df["tgt"], tokenizer)
    n_train = int(train_split * len(full_ds))
    n_val   = len(full_ds) - n_train
    train_ds, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
    kw = dict(num_workers=num_workers,
              pin_memory=torch.cuda.is_available(),
              collate_fn=_collate)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  drop_last=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, drop_last=False, **kw)
    return train_loader, val_loader


def make_lazy_loader(df, tokenizer, batch_size, num_workers, shuffle=False):
    ds = LazyTranslationDataset(df["src"], df["tgt"], tokenizer)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers,
                      pin_memory=torch.cuda.is_available(),
                      drop_last=False, collate_fn=_collate)


pretrain_train_loader, pretrain_val_loader = make_lazy_loaders(
    pretrain_df, tokenizer,
    CFG["train_split"], CFG["batch_size"], CFG["num_workers"])

finetune_train_loader, finetune_val_loader = make_lazy_loaders(
    finetune_train_df, tokenizer,
    CFG["train_split"], CFG["batch_size"], CFG["num_workers"])

finetune_test_loader = make_lazy_loader(
    finetune_test_df, tokenizer,
    CFG["batch_size"], CFG["num_workers"])

print(f"Pretrain  — train: {len(pretrain_train_loader)} | val: {len(pretrain_val_loader)}")
print(f"Finetune  — train: {len(finetune_train_loader)} | val: {len(finetune_val_loader)}")
print(f"Finetune  — test:  {len(finetune_test_loader)}")

# DataFrames can be freed now — the lists are already copied into the Datasets
del finetune_test_df, finetune_train_df, pretrain_df, ft_df
gc.collect()

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
 
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]
 
 
class LoRALinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with a trainable low-rank delta.
 
    Forward pass:
        y = x W^T + b  +  (dropout(x) A^T) B^T * (alpha / r)
              ^^^^^^^^^^^    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              frozen base              trainable LoRA delta
 
    Initialisation:
        A ~ Kaiming-uniform  (so the delta is non-trivially shaped from step 1)
        B  = 0               (so the delta starts at exactly zero → same as pretrained)
 
    After fine-tuning call .merge() to fold the delta into W in-place.
    The merged layer behaves identically to a plain nn.Linear at inference.
    """
    def __init__(self, in_features: int, out_features: int,
                 r: int, lora_alpha: float, lora_dropout: float = 0.0,
                 bias: bool = True):
        super().__init__()
        assert r > 0, "LoRA rank r must be a positive integer"
        self.in_features  = in_features
        self.out_features = out_features
        self.r            = r
        self.scaling      = lora_alpha / r      # pre-compute the constant scale
 
        # ── Frozen base weight (same layout as nn.Linear) ─────────────────────
        self.weight = nn.Parameter(torch.empty(out_features, in_features),
                                   requires_grad=False)
        self.bias   = nn.Parameter(torch.zeros(out_features),
                                   requires_grad=False) if bias else None
 
        # ── Trainable LoRA matrices ────────────────────────────────────────────
        # A: shape (r, in)  — projects input down to rank-r subspace
        # B: shape (out, r) — projects back up; init to zero so delta starts at 0
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
 
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
        self._merged = False
        self._reset_base_weights()
 
    def _reset_base_weights(self):
        """Initialise W and b exactly as nn.Linear does."""
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)
        # A gets Kaiming so gradients flow from the very first step;
        # B stays zero so the initial model output is identical to the pretrained one.
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base linear (frozen)
        out = F.linear(x, self.weight, self.bias)
        if not self._merged:
            # LoRA delta: x → (r,) → (out,)
            lora_out = F.linear(self.lora_dropout(x), self.lora_A)   # (..., r)
            lora_out = F.linear(lora_out, self.lora_B)                # (..., out)
            out = out + lora_out * self.scaling
        return out
 
    def merge(self):
        """
        Fold the LoRA delta into W in-place.
        After merging, forward() is identical to a plain nn.Linear — zero overhead.
        """
        if self._merged:
            return
        # delta W = B @ A,  shape (out, in) — same as self.weight
        with torch.no_grad():
            self.weight.data += (self.lora_B @ self.lora_A) * self.scaling
        self._merged = True
 
    def extra_repr(self) -> str:
        return (f"in={self.in_features}, out={self.out_features}, "
                f"r={self.r}, scaling={self.scaling:.3f}, merged={self._merged}")
 
 
class AttentionLayer(nn.Module):
    """Single attention head. Optionally wraps Q/K/V projections with LoRA."""
    def __init__(self, d_model, d_k, d_v, lora_cfg=None):
        super().__init__()
        if lora_cfg:
            r, alpha, drop = lora_cfg["r"], lora_cfg["alpha"], lora_cfg["dropout"]
            self.W_q = LoRALinear(d_model, d_k, r=r, lora_alpha=alpha, lora_dropout=drop)
            self.W_k = LoRALinear(d_model, d_k, r=r, lora_alpha=alpha, lora_dropout=drop)
            self.W_v = LoRALinear(d_model, d_v, r=r, lora_alpha=alpha, lora_dropout=drop)
        else:
            self.W_q = nn.Linear(d_model, d_k)
            self.W_k = nn.Linear(d_model, d_k)
            self.W_v = nn.Linear(d_model, d_v)
 
    def forward(self, q, k, v, mask=None):
        Q, K, V = self.W_q(q), self.W_k(k), self.W_v(v)
        scores = (Q @ K.transpose(-1, -2)) / math.sqrt(K.size(-1))
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        return torch.softmax(scores, dim=-1) @ V
 
 
class AdapterLayer(nn.Module):
    """
    Houlsby-style bottleneck adapter (Houlsby et al., 2019).
 
    Architecture (applied after each sub-layer's residual + LayerNorm):
        h = x + W_up( GELU( W_down(x) ) )
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              trainable bottleneck  (d_model → r → d_model)
 
    All original transformer weights are frozen; only W_down, W_up, and their
    biases are trained. This is architecturally different from LoRA:
      - LoRA injects a delta *inside* existing weight matrices (W += BA·scale)
      - Adapters add *new* sub-networks in between existing layers
 
    Initialisation:
        W_down ~ N(0, 0.01)   small but non-zero so training starts immediately
        W_up   = 0            output starts at zero → residual = identity at step 0
    """
    def __init__(self, d_model: int, bottleneck: int, dropout: float = 0.0):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck)
        self.up      = nn.Linear(bottleneck, d_model)
        self.act     = nn.GELU()
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self._init_weights()
 
    def _init_weights(self):
        # Small normal init for down — gives useful gradients from step 1
        nn.init.normal_(self.down.weight, std=0.01)
        nn.init.zeros_(self.down.bias)
        # Zero init for up — residual path starts as identity
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.up(self.dropout(self.act(self.down(x))))
 
    def extra_repr(self) -> str:
        return f"d_model→{self.down.out_features}→{self.up.out_features}"
 
 
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, lora_cfg=None):
        super().__init__()
        self.heads = nn.ModuleList(
            [AttentionLayer(d_model, d_k, d_v, lora_cfg) for _ in range(n_heads)])
        self.lin = nn.Linear(n_heads * d_v, d_model)
 
    def forward(self, q, k, v, mask=None):
        return self.lin(torch.cat([h(q, k, v, mask) for h in self.heads], dim=-1))
 
 
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
 
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))
 
 
class EncoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, d_ff, lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads, d_model, d_k, d_v, lora_cfg)
        self.ffn = FeedForward(d_model, d_ff)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        # Houlsby: one adapter after attention sub-layer, one after FFN sub-layer
        if adapter_cfg:
            b, drop = adapter_cfg["bottleneck"], adapter_cfg["dropout"]
            self.adapter_attn = AdapterLayer(d_model, b, drop)
            self.adapter_ffn  = AdapterLayer(d_model, b, drop)
        else:
            self.adapter_attn = None
            self.adapter_ffn  = None
 
    def forward(self, x, src_mask=None):
        x = self.ln1(x + self.mha(x, x, x, src_mask))
        if self.adapter_attn is not None:
            x = self.adapter_attn(x)
        x = self.ln2(x + self.ffn(x))
        if self.adapter_ffn is not None:
            x = self.adapter_ffn(x)
        return x
 
 
class DecoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, d_ff, lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.self_attn  = MultiHeadAttention(n_heads, d_model, d_k, d_v, lora_cfg)
        self.cross_attn = MultiHeadAttention(n_heads, d_model, d_k, d_v, lora_cfg)
        self.ffn  = FeedForward(d_model, d_ff)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.ln3  = nn.LayerNorm(d_model)
        # Houlsby: adapter after self-attn, after cross-attn, and after FFN
        if adapter_cfg:
            b, drop = adapter_cfg["bottleneck"], adapter_cfg["dropout"]
            self.adapter_self  = AdapterLayer(d_model, b, drop)
            self.adapter_cross = AdapterLayer(d_model, b, drop)
            self.adapter_ffn   = AdapterLayer(d_model, b, drop)
        else:
            self.adapter_self  = None
            self.adapter_cross = None
            self.adapter_ffn   = None
 
    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        tgt = self.ln1(tgt + self.self_attn(tgt, tgt, tgt, tgt_mask))
        if self.adapter_self is not None:
            tgt = self.adapter_self(tgt)
        tgt = self.ln2(tgt + self.cross_attn(tgt, memory, memory, src_mask))
        if self.adapter_cross is not None:
            tgt = self.adapter_cross(tgt)
        tgt = self.ln3(tgt + self.ffn(tgt))
        if self.adapter_ffn is not None:
            tgt = self.adapter_ffn(tgt)
        return tgt
 
 
class Encoder(nn.Module):
    def __init__(self, vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                 lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.layers  = nn.ModuleList(
            [EncoderLayer(n_heads, d_model, d_k, d_v, d_ff, lora_cfg, adapter_cfg)
             for _ in range(n_layers)])
 
    def forward(self, x, src_mask=None):
        x = self.pos_enc(self.embed(x))
        for layer in self.layers:
            x = layer(x, src_mask)
        return x
 
 
class Decoder(nn.Module):
    def __init__(self, vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                 lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.layers  = nn.ModuleList(
            [DecoderLayer(n_heads, d_model, d_k, d_v, d_ff, lora_cfg, adapter_cfg)
             for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size)
 
    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        x = self.pos_enc(self.embed(tgt))
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, src_mask)
        return self.out(x)
 
 
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size,
                 n_heads, d_model, d_k, d_v, d_ff, n_layers,
                 lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                                lora_cfg, adapter_cfg)
        self.decoder = Decoder(tgt_vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                                lora_cfg, adapter_cfg)
 
    def _causal_mask(self, sz, dev):
        return ~torch.triu(torch.ones(sz, sz, device=dev, dtype=torch.bool), diagonal=1)
 
    def forward(self, src, tgt):
        tgt_mask = self._causal_mask(tgt.size(1), src.device)
        memory   = self.encoder(src)
        return self.decoder(tgt, memory, tgt_mask=tgt_mask)

In [ ]:
 
def build_lora_cfg(cfg: dict) -> dict:
    return {"r": cfg["lora_r"], "alpha": cfg["lora_alpha"], "dropout": cfg["lora_dropout"]}
 
 
def mark_only_lora_as_trainable(model: nn.Module) -> None:
    """
    Freeze every parameter except LoRA matrices.
    A parameter is considered a LoRA parameter iff its name ends in
    'lora_A' or 'lora_B' — the naming convention used by LoRALinear.
    """
    for name, param in model.named_parameters():
        param.requires_grad = name.endswith(("lora_A", "lora_B"))
 
 
def merge_lora_weights(model: nn.Module) -> None:
    """
    Walk the model and call .merge() on every LoRALinear module.
    After this call the model is equivalent to a plain Transformer —
    no LoRA overhead at inference, and it can be saved as a normal checkpoint.
    """
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module.merge()
 
 
def enable_lora(model: nn.Module) -> nn.Module:
    """
    Freeze all non-LoRA weights and report trainable parameter count.
    Call AFTER transferring pretrained weights into the LoRA model.
    """
    mark_only_lora_as_trainable(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"LoRA trainable params: {trainable:,} / {total:,}  "
          f"({100 * trainable / total:.2f} %)")
    return model
 
 
def transfer_weights_to_lora_model(pretrained: nn.Module, lora_model: nn.Module) -> nn.Module:
    """
    Copy all matching base weights from a plain model into a LoRA model.
 
    Keys that exist only in the LoRA model (lora_A, lora_B) are intentionally
    skipped so they keep their initialised values:
      - lora_B stays zero  → delta is zero at the start of fine-tuning
      - lora_A stays Kaiming → gradients flow from step 1
    """
    pretrained_sd = pretrained.state_dict()
    lora_sd       = lora_model.state_dict()
 
    matched, skipped = 0, 0
    for key in lora_sd:
        if key in pretrained_sd and lora_sd[key].shape == pretrained_sd[key].shape:
            lora_sd[key] = pretrained_sd[key]
            matched += 1
        else:
            skipped += 1   # lora_A / lora_B keys — leave at init values
 
    lora_model.load_state_dict(lora_sd)
    print(f"Weight transfer: {matched} matched, {skipped} skipped (LoRA deltas stay at init)")
    return lora_model

In [ ]:
def build_adapter_cfg(cfg: dict) -> dict:
    return {"bottleneck": cfg["adapter_bottleneck"], "dropout": cfg["adapter_dropout"]}
 
 
def mark_only_adapters_as_trainable(model: nn.Module) -> None:
    """
    Freeze every parameter except AdapterLayer weights.
    All original transformer weights (embeddings, attention, FFN, LN) stay frozen.
    """
    for name, param in model.named_parameters():
        # AdapterLayer sub-modules are named adapter_attn/adapter_ffn/adapter_self/adapter_cross
        # Their parameters contain 'adapter_' in the full dotted path
        param.requires_grad = "adapter_" in name
 
 
def enable_adapters(model: nn.Module) -> nn.Module:
    """Freeze everything except adapter parameters; report trainable count."""
    mark_only_adapters_as_trainable(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Adapter trainable params: {trainable:,} / {total:,}  "
          f"({100 * trainable / total:.2f} %)")
    return model
 
 
def transfer_weights_to_adapter_model(pretrained: nn.Module,
                                       adapter_model: nn.Module) -> nn.Module:
    """
    Copy all matching base weights from a plain pretrained model into an
    adapter model. AdapterLayer parameters (down/up projections) are not
    present in the pretrained dict, so they keep their zero initialisation.
    """
    pretrained_sd  = pretrained.state_dict()
    adapter_sd     = adapter_model.state_dict()
 
    matched, skipped = 0, 0
    for key in adapter_sd:
        if key in pretrained_sd and adapter_sd[key].shape == pretrained_sd[key].shape:
            adapter_sd[key] = pretrained_sd[key]
            matched += 1
        else:
            skipped += 1   # adapter down/up weights — keep zero init
 
    adapter_model.load_state_dict(adapter_sd)
    print(f"Weight transfer: {matched} matched, {skipped} skipped (adapter weights stay at init)")
    return adapter_model

In [ ]:

def evaluate(model, loader, criterion, desc="Val"):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    
    all_preds = []
    all_targets = []
    
    bar = tqdm.tqdm(loader, dynamic_ncols=True, leave=False, desc=desc)
    
    with torch.no_grad():
        for src, tgt in bar:
            src = src.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            tgt_in = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            with torch.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                logits = model(src, tgt_in)
                logits_flat = logits.reshape(-1, logits.size(-1))
                tgt_flat = tgt_out.reshape(-1)
                loss = criterion(logits_flat, tgt_flat)

            total_loss += loss.item()
            
            # --- Accuracy Calculation ---
            preds_ids = logits.argmax(dim=-1) # Shape: [batch, seq_len]
            mask = tgt_out != criterion.ignore_index
            total_correct += ((preds_ids == tgt_out) & mask).sum().item()
            total_tokens += mask.sum().item()

            # --- BLEU Preparation ---
            # Convert IDs to strings for each sentence in the batch
            for i in range(tgt_out.size(0)):
                # Get predicted and target IDs
                p_ids = preds_ids[i].tolist()
                t_ids = tgt_out[i].tolist()
                

                all_preds.append(tokenizer.decode(p_ids, skip_special_tokens=True))
                all_targets.append(tokenizer.decode(t_ids, skip_special_tokens=True))

    # Calculate final metrics
    avg_loss = total_loss / max(len(loader), 1)
    avg_acc = total_correct / max(total_tokens, 1)
    
    # BLEU expects a list of lists for references: [ [ref1, ref2...], [ref1_b, ref2_b...] ]
    bleu_score = sacrebleu.corpus_bleu(all_preds, [all_targets]).score

    return avg_loss, avg_acc, bleu_score
def train_epoch(model, loader, optimizer, criterion, scaler, scheduler=None, grad_accum_steps=1, desc="Train"):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    optimizer.zero_grad(set_to_none=True)
    bar = tqdm.tqdm(enumerate(loader), total=len(loader), dynamic_ncols=True, leave=False, desc=desc)
    print(bar)
    for step, (src, tgt) in bar:
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        with torch.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            logits = model(src, tgt_in)
            logits_flat = logits.reshape(-1, logits.size(-1))
            tgt_flat = tgt_out.reshape(-1)
            raw_loss = criterion(logits_flat, tgt_flat)
            loss = raw_loss / grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            # --- Scheduler Step ---
            if scheduler is not None:
                scheduler.step()
                
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item()
        preds = logits_flat.argmax(dim=-1)
        mask = tgt_flat != criterion.ignore_index
        total_correct += ((preds == tgt_flat) & mask).sum().item()
        total_tokens += mask.sum().item()

        bar.set_postfix(loss=f"{raw_loss.item():.4f}")

    return total_loss / max(len(loader), 1), total_correct / max(total_tokens, 1)
    

def run_training(model, train_loader, val_loader, optimizer, criterion, scaler, scheduler, epochs, grad_accum_steps=1, stage_name="stage"):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_bleu": []}
    
    for epoch in range(epochs):
        # 1. Training Phase (Passes scheduler here)
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, scaler,
            scheduler=scheduler, 
            grad_accum_steps=grad_accum_steps, 
            desc=f"{stage_name} train {epoch+1}"
        )
        
        # 2. Evaluation Phase
        val_loss, val_acc, val_bleu = evaluate(
            model, val_loader, criterion, desc=f"{stage_name} val {epoch+1}"
        )
        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_bleu"].append(val_bleu)
        
        print(f"{stage_name} Epoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val BLEU: {val_bleu:.2f}")
        # Optional: Print current LR
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  End LR:     {current_lr:.6f}")
        print("-" * 30)
        
    return history

In [ ]:

pretrain_model = Transformer(
    src_vocab_size=CFG["vocab_size"],
    tgt_vocab_size=CFG["vocab_size"],
    n_heads=CFG["n_heads"],
    d_model=CFG["d_model"],
    d_k=CFG["d_k"],
    d_v=CFG["d_v"],
    d_ff=CFG["d_ff"],
    n_layers=CFG["n_layers"],
).to(device)

pretrain_optimizer = optim.AdamW(pretrain_model.parameters(), lr=CFG["lr_pretrain"], betas=(0.9, 0.98), eps=1e-9)
pretrain_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
pretrain_scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
total_steps = (len(pretrain_train_loader) // CFG["grad_accum_steps"]) * CFG["pretrain_epochs"]

pretrain_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    pretrain_optimizer,
    max_lr=CFG["lr_pretrain"],
    total_steps=total_steps,
    pct_start=0.1, # 10% warmup
    anneal_strategy='cos'
)

sum(p.numel() for p in pretrain_model.parameters())


In [ ]:
gc.collect()
pretrain_history = run_training(
    model=pretrain_model,
    train_loader=pretrain_train_loader,
    val_loader=pretrain_val_loader,
    optimizer=pretrain_optimizer,
    criterion=pretrain_criterion,
    scaler=pretrain_scaler,
    scheduler=pretrain_scheduler,
    epochs=CFG["pretrain_epochs"],
    grad_accum_steps=CFG["grad_accum_steps"],
    stage_name="pretrain en-fr",
)

print("\n=== Baseline: pretrained model on Juridia val (no fine-tuning) ===")
baseline_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
baseline_loss, baseline_acc, baseline_bleu = evaluate(
    pretrain_model, finetune_val_loader, baseline_criterion,
    desc="baseline eval")
print(f"  Loss: {baseline_loss:.4f} | Acc: {baseline_acc:.4f} | BLEU: {baseline_bleu:.2f}")

In [ ]:
# Save pretrained checkpoint
PRETRAIN_CKPT = "/home/ubuntu/10701/pretrained_fr_en_transformer.pt"
torch.save({
    "model_state_dict": pretrain_model.state_dict(),
    "config": CFG,
}, PRETRAIN_CKPT)
print("Saved pretrained checkpoint:", PRETRAIN_CKPT)


import wandb
import os

# Fetch the secret
wandb_key = os.getenv("WANDB_API_KEY")

# Login
wandb.login(key=wandb_key)

wandb.init(project="transformer-translation", config=CFG)

wandb.log({
    "pretrain_final_loss": pretrain_history["train_loss"][-1],
    "pretrain_final_val_loss": pretrain_history["val_loss"][-1],
})

# Save checkpoint
PRETRAIN_CKPT = "/home/ubuntu/10701/pretrained_fr_en_transformer.pt"
torch.save({
    "model_state_dict": pretrain_model.state_dict(),
    "config": CFG,
}, PRETRAIN_CKPT)

# Log artifact
artifact = wandb.Artifact(
    name="fr-en-transformer-pretrained",
    type="model",
    description="Pretrained FR→EN Transformer"
)
artifact.add_file(PRETRAIN_CKPT)
wandb.log_artifact(artifact)

print("Logged pretrained model to W&B")

In [ ]:
lora_cfg   = build_lora_cfg(CFG)
 
lora_model = Transformer(
    src_vocab_size=CFG["vocab_size"],
    tgt_vocab_size=CFG["vocab_size"],
    n_heads=CFG["n_heads"],  d_model=CFG["d_model"],
    d_k=CFG["d_k"],          d_v=CFG["d_v"],
    d_ff=CFG["d_ff"],         n_layers=CFG["n_layers"],
    lora_cfg=lora_cfg,
).to(device)
 
lora_model = transfer_weights_to_lora_model(pretrain_model, lora_model)
lora_model = enable_lora(lora_model)
 
lora_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, lora_model.parameters()),
    lr=CFG["lr_finetune"], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
lora_criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab["<pad>"])
lora_scaler    = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
 
total_lora_steps = (len(finetune_train_loader) // CFG["grad_accum_steps"]) * CFG["finetune_epochs"]
lora_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    lora_optimizer, max_lr=CFG["lr_finetune"],
    total_steps=total_lora_steps, pct_start=0.1, anneal_strategy='cos')
 
lora_history = run_training(
    model=lora_model,
    train_loader=finetune_train_loader,
    val_loader=finetune_val_loader,
    optimizer=lora_optimizer,
    criterion=lora_criterion,
    scaler=lora_scaler,
    scheduler=lora_scheduler,
    epochs=CFG["finetune_epochs"],
    grad_accum_steps=CFG["grad_accum_steps"],
    stage_name="finetune LoRA fr→en",
)
merge_lora_weights(lora_model)


In [ ]:
LORA_CKPT = "/home/ubuntu/10701/finetuned_lora_fr_en_transformer.pt"
torch.save({
    "model_state_dict": lora_model.state_dict(),
    "config": CFG,
}, LORA_CKPT)
print("Saved LoRA fine-tuned checkpoint:", LORA_CKPT)

In [ ]:
adapter_cfg = build_adapter_cfg(CFG)
 
adapter_model = Transformer(
    src_vocab_size=CFG["vocab_size"],
    tgt_vocab_size=CFG["vocab_size"],
    n_heads=CFG["n_heads"],  d_model=CFG["d_model"],
    d_k=CFG["d_k"],          d_v=CFG["d_v"],
    d_ff=CFG["d_ff"],         n_layers=CFG["n_layers"],
    adapter_cfg=adapter_cfg,
).to(device)
 
adapter_model = transfer_weights_to_adapter_model(pretrain_model, adapter_model)
adapter_model = enable_adapters(adapter_model)
 
adapter_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter_model.parameters()),
    lr=CFG["lr_adapter"], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
adapter_criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab["<pad>"])
adapter_scaler    = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
 
total_adapter_steps = (len(finetune_train_loader) // CFG["grad_accum_steps"]) * CFG["adapter_epochs"]
adapter_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    adapter_optimizer, max_lr=CFG["lr_adapter"],
    total_steps=total_adapter_steps, pct_start=0.1, anneal_strategy='cos')
 
adapter_history = run_training(
    model=adapter_model,
    train_loader=finetune_train_loader,
    val_loader=finetune_val_loader,
    optimizer=adapter_optimizer,
    criterion=adapter_criterion,
    scaler=adapter_scaler,
    scheduler=adapter_scheduler,
    epochs=CFG["adapter_epochs"],
    grad_accum_steps=CFG["grad_accum_steps"],
    stage_name="finetune Adapter fr→en",
)
 
ADAPTER_CKPT = "/home/ubuntu/10701/finetuned_adapter_fr_en_transformer.pt"
torch.save({
    "model_state_dict": adapter_model.state_dict(),
    "config": CFG,
}, ADAPTER_CKPT)
print("Saved adapter fine-tuned checkpoint:", ADAPTER_CKPT)

In [ ]:

def translate_sentence(model, sentence, src_vocab, tgt_vocab, inv_tgt_vocab, max_len):
    model.eval()
    sentence = clean_text(sentence)
    src_ids = encode_and_pad(sentence.split(), src_vocab, max_len)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    generated = [tgt_vocab["<bos>"]]
    with torch.no_grad():
        for _ in range(max_len - 1):
            tgt = torch.tensor([generated], dtype=torch.long, device=device)
            logits = model(src, tgt)
            next_id = logits[0, -1].argmax().item()
            generated.append(next_id)
            if next_id == tgt_vocab["<eos>"]:
                break

    words = []
    for idx in generated[1:]:
        if idx == tgt_vocab["<eos>"]:
            break
        if idx not in {tgt_vocab["<pad>"], tgt_vocab["<bos>"]}:
            words.append(inv_tgt_vocab.get(idx, "<unk>"))
    return " ".join(words)

test_sentence = "the weather is very nice today ."
translate_sentence(finetune_model, test_sentence, src_vocab, de_vocab, inv_de_vocab, CFG["max_len"])


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(pretrain_history["train_loss"], label="pretrain train")
plt.plot(pretrain_history["val_loss"], label="pretrain val")
plt.plot(finetune_history["train_loss"], label="finetune train")
plt.plot(finetune_history["val_loss"], label="finetune val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
